### Install the dependencies

In [ ]:
!pip install spacy networkx matplotlib graphviz pygraphviz

### Install spacy model

In [ ]:
!python -m spacy download "en_core_web_trf"

## GRAPHER

> Write the sentence you wish to graph in the pop up.

In [ ]:
import spacy
import networkx as nx
import matplotlib.pyplot as plt

nlp = spacy.load("en_core_web_trf")

def visualize_dependency_graph(text):
    """Visualizes the dependency graph of a given text, ensuring unique nodes for each token."""
    doc = nlp(text)

    G = nx.DiGraph()
    root_node = None

    # Build graph with unique node IDs using token.i
    for token in doc:
        if token.pos_ != "PUNCT":
            node_id = token.i  # Unique identifier for each token
            G.add_node(
                node_id,
                token=token.text,
                pos=token.pos_,
                tag=token.tag_,
                dep=token.dep_
            )
            if token.dep_ == "ROOT":
                root_node = node_id
            if token.dep_ != "ROOT":
                G.add_edge(token.head.i, token.i, dep=token.dep_)

    try:
        pos = nx.nx_agraph.graphviz_layout(G, prog="dot")
    except ImportError:
        print("pygraphviz is not installed, install it or update networkx")
        pos = nx.spring_layout(G)

    edge_labels = nx.get_edge_attributes(G, "dep")

    # Prepare node labels and colors
    node_labels = {}
    node_colors = []
    
    for node, data in G.nodes(data=True):
        label_str = f"({data['token']})\nPOS: {data['pos']}\nTAG: {data['tag']}"
        node_labels[node] = label_str
        
        # Choose colors based on dependency and part-of-speech
        if data["dep"] == "attr":
            color = "#b0c4b1"
        elif data["pos"] == "VERB":
            color = "lightgreen"
        elif data["pos"] == "ADJ":
            color = "lightpink"
        elif data["pos"] == "DET":
            color = "peachpuff"
        elif data["pos"] in ["PROPN", "NOUN"]:
            color = "plum"
        else:
            color = "lightblue"
            
        node_colors.append(color)

    # Highlight the root node if it exists
    if root_node in G.nodes():
        # Find the index of the root node in the list of nodes
        root_index = list(G.nodes()).index(root_node)
        node_colors[root_index] = "lightcoral"

    plt.figure(figsize=(12, 8))
    plt.text(
        0.01,
        0.95,
        f"Statement: {text}",
        transform=plt.gcf().transFigure,
        fontsize=10,
        verticalalignment="top",
        horizontalalignment="left"
    )
    nx.draw(
        G,
        pos,
        labels=node_labels,
        with_labels=True,
        node_size=3500,
        node_color=node_colors,
        arrowsize=18,
        font_size=9
    )
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8)

    plt.title("Dependency Tree (Graphviz Layout, Colored Nodes)")
    plt.show()

# Example usage
example_text = input("Enter your statement: ")
visualize_dependency_graph(example_text)
print(example_text, '\n')

doc = nlp(example_text)
print(f"MORPH\t:\tANCESTORS")
for token in doc:
    print(f"{token} : {token.morph} : {list(token.ancestors)}")




### Latex Generator

In [ ]:
import spacy
import math

# Load spaCy model
try:
    nlp = spacy.load("en_core_web_trf")
except OSError:
    print("Downloading en_core_web_trf model...")
    spacy.cli.download("en_core_web_trf")
    nlp = spacy.load("en_core_web_trf")

# --- TikZ Generation Configuration ---
# COLOR_MAP uses SVG color names, requiring [svgnames] option for xcolor.
# Capitalization matters for SVG names.
COLOR_MAP = {
    "ROOT_NODE_COLOR": "LightCoral",
    "VERB": "LightGreen",
    "ADJ": "LightPink",
    "DET": "PeachPuff",
    "PROPN": "Plum",
    "NOUN": "Plum",
    "ATTR_DEP_COLOR": "LimeGreen",
    "DEFAULT_NODE_COLOR": "LightBlue"
}

# LaTeX special characters that need escaping
LATEX_ESCAPE_CHARS = {
    '&': r'\&',
    '%': r'\%',
    '$': r'\$',
    '#': r'\#',
    '_': r'\_',
    '{': r'\{',
    '}': r'\}',
    '~': r'\textasciitilde{}',
    '^': r'\textasciicircum{}',
    '\\': r'\textbackslash{}',
}

def escape_latex(text):
    """Escapes special LaTeX characters in a string."""
    return "".join(LATEX_ESCAPE_CHARS.get(char, char) for char in text)

def get_node_color(token, is_root):
    """Determines the TikZ fill color for a token."""
    if is_root:
        return COLOR_MAP["ROOT_NODE_COLOR"]
    if token.dep_.lower() == "attr": # Check for 'attr' dependency
         return COLOR_MAP["ATTR_DEP_COLOR"]
    return COLOR_MAP.get(token.pos_, COLOR_MAP["DEFAULT_NODE_COLOR"])

# Global lists to store TikZ commands
node_definitions_tikz = []
edge_definitions_tikz = []
processed_tokens_for_nodes = set() # To avoid duplicate node definitions

def generate_tikz_recursive(doc, token_idx, parent_tikz_id=None, level=0, sibling_idx=0, num_siblings=0):
    """
    Recursively generates TikZ node and edge definitions.
    Uses token_idx (spaCy token.i) for tracking.
    """
    current_token = doc[token_idx]
    current_tikz_id = f"t{current_token.i}"

    if current_token.i in processed_tokens_for_nodes:
        if parent_tikz_id:
            dep_label = escape_latex(current_token.dep_)
            edge_label_opts = "auto" 
            if num_siblings > 0 :
                angle_approx = (sibling_idx - (num_siblings - 1) / 2.0)
                if abs(angle_approx) < 0.1 : 
                    edge_label_opts = "right"
                elif angle_approx < 0: 
                    edge_label_opts = "above"
                else: 
                    edge_label_opts = "above"
            edge_definitions_tikz.append(f"\\draw[edge style] ({parent_tikz_id}) -- ({current_tikz_id}) node[label style, {edge_label_opts}] {{{dep_label}}};")
        return

    processed_tokens_for_nodes.add(current_token.i)

    node_text_label = escape_latex(current_token.text)
    pos_label = escape_latex(current_token.pos_)
    tag_label = escape_latex(current_token.tag_)
    # Make the main token text Large and bold
    label_content = f"{{\\small\\textbf{{({node_text_label})}}}}\\\\POS: {pos_label}\\\\TAG: {tag_label}"
    
    is_root_node = current_token.dep_ == "ROOT"
    color = get_node_color(current_token, is_root_node)
    
    node_options_list = [f"node style, fill={color}"]

    x_offset_val_for_edge_label = 0 # Initialize for root or single child case

    if parent_tikz_id is None: 
        node_definitions_tikz.append(f"\\node ({current_tikz_id}) [{', '.join(node_options_list)}] {{{label_content}}};")
    else:
        y_distance = "1.8cm"  # Adjusted y_distance for potentially larger nodes
        if num_siblings == 1:
            position_keyword = "below"
            dist_str = y_distance
        else:
            mid_point = (num_siblings - 1) / 2.0
            horizontal_spacing_factor = 1.5  # Adjusted horizontal_spacing_factor
            x_offset_val_for_edge_label = (sibling_idx - mid_point) * horizontal_spacing_factor # Store for edge label
            
            if abs(x_offset_val_for_edge_label) < 0.1: 
                position_keyword = "below"
                dist_str = y_distance
            elif x_offset_val_for_edge_label < 0: 
                position_keyword = "below left"
                dist_str = f"{y_distance} and {abs(x_offset_val_for_edge_label):.1f}cm"
            else: 
                position_keyword = "below right"
                dist_str = f"{y_distance} and {abs(x_offset_val_for_edge_label):.1f}cm"
        
        node_options_list.append(f"{position_keyword}=of {parent_tikz_id}, node distance={dist_str}")
        node_definitions_tikz.append(f"\\node ({current_tikz_id}) [{', '.join(node_options_list)}] {{{label_content}}};")

        dep_label = escape_latex(current_token.dep_)
        edge_label_opts = "auto" 
        if abs(x_offset_val_for_edge_label) < 0.1 : edge_label_opts = "right"
        elif x_offset_val_for_edge_label < 0 : edge_label_opts = "sloped,above"
        else: edge_label_opts = "sloped,above"
        edge_definitions_tikz.append(f"\\draw[edge style] ({parent_tikz_id}) -- ({current_tikz_id}) node[label style, {edge_label_opts}] {{{dep_label}}};")

    children_indices = [c.i for c in current_token.children if c.pos_ != "PUNCT"]
    num_children = len(children_indices)
    for i, child_idx in enumerate(children_indices):
        generate_tikz_recursive(doc, child_idx, current_tikz_id, level + 1, i, num_children)


def spacy_to_tikz(text):
    global node_definitions_tikz, edge_definitions_tikz, processed_tokens_for_nodes
    node_definitions_tikz = []
    edge_definitions_tikz = []
    processed_tokens_for_nodes = set()

    doc = nlp(text)
    root_indices = [token.i for token in doc if token.dep_ == "ROOT" and token.pos_ != "PUNCT"]

    if not root_indices:
        first_token_idx = next((token.i for token in doc if token.pos_ != "PUNCT"), None)
        if first_token_idx is not None:
            root_indices = [first_token_idx]
        else:
            return "% No processable tokens found in text."

    generate_tikz_recursive(doc, root_indices[0])
    
    for token in doc:
        if token.pos_ != "PUNCT" and token.i not in processed_tokens_for_nodes:
            # Make the main token text Large and bold for unconnected nodes too
            temp_label = f"{{\\small\\textbf{{({escape_latex(token.text)})}}}}\\\\POS: {escape_latex(token.pos_)}\\\\TAG: {escape_latex(token.tag_)}" 
            temp_color = get_node_color(token, token.dep_ == "ROOT")
            unconnected_y_offset = len(doc) * 1.2 + len(processed_tokens_for_nodes) * 0.4 
            node_definitions_tikz.append(f"% Unconnected Node - manual placement might be needed:\n\\node (t{token.i}) [node style, fill={temp_color}, at={{(0,-{unconnected_y_offset}cm)}}] {{{temp_label}}};")
            processed_tokens_for_nodes.add(token.i)
            if token.head and token.head.i in processed_tokens_for_nodes and token.head.i != token.i:
                 dep_label = escape_latex(token.dep_)
                 edge_definitions_tikz.append(f"\\draw[edge style] (t{token.head.i}) -- (t{token.i}) node[label style, auto] {{{dep_label}}};")

    latex_preamble = f"""
\\begin{{figure}}[H]
\\begin{{center}} % Center the TikZ picture
\\scalebox{{.75}}{{
\\begin{{tikzpicture}}[
    node distance = 2cm and 1.5cm, % Adjusted node distance for larger nodes
    node style/.style = {{
        circle, 
        draw,
        thick,
        fill=white, 
        align=center,
        minimum size=2cm, % Increased minimum size for circles to fit Large text
        inner sep=1pt, 
        font=\\footnotesize % Smaller font for POS/TAG lines
    }},
	ex style/.style = {{
		draw=black!50,
		dashed,
		shorten >=1pt,
		shorten <=1pt
	}},
	ex arc/.style={{
		draw=black!50,
		dashed,
		shorten >=1pt,
		shorten <=1pt
	}},
    edge style/.style = {{
        draw,
        thick,
        -Latex 
    }},
    label style/.style = {{
        midway,
        font=\\small, % Edge labels
        fill=white, 
        inner sep=1pt 
    }}
]

% Node Definitions
"""
    
    nodes_str = "\n".join(node_definitions_tikz)
    edges_str = "\n".join(edge_definitions_tikz)
    
    latex_document = latex_preamble + nodes_str + "\n\n% Edge Definitions\n" + edges_str + \
                     "\n\n\\end{tikzpicture}}\n\\end{center} % End center environment\n\\end{figure}"
    
    return latex_document

if __name__ == "__main__":
    example_text = input("Enter your statement: ")
    
    if example_text:
        tikz_output = spacy_to_tikz(example_text)
        print(tikz_output)
        
        # To save the output to a .tex file:
        # with open("dependency_tree.tex", "w") as f:
        #     f.write(tikz_output)
        # print("\nOutput saved to dependency_tree.tex")
    else:
        print("No input provided.")

